In [1]:
from google.cloud import bigquery
from datetime import datetime, timedelta
import random

In [2]:
project_id = 'pcloud2026'
region = 'europe-west8'
db_id = 'ais2023'
table_id = 'ais2023'
client = bigquery.Client.from_service_account_json('../secret.json')

In [7]:
# query bigqeury to group by MMSI and compute the maximum delta of LAT in each group
#order from the highest to the lowest
#limit 100
querys = f"""
    SELECT MMSI, MAX(LAT) - MIN(LAT) AS delta_lat
    FROM `{project_id}.{db_id}.{table_id}`
    GROUP BY MMSI
    ORDER BY delta_lat DESC
    LIMIT 100
    """ 
df = client.query(querys).to_dataframe()
df

,MMSI,delta_lat
0,367354090,62.37766
1,338209606,60.89126
2,366933870,57.00005
3,338424122,55.92408
4,368281980,42.66238
...,...,...
95,566167000,23.91041
96,636092932,23.86376
97,229395000,23.74333
98,374077000,23.64844


In [8]:
querys = f"""
    SELECT *
    FROM `{project_id}.{db_id}.{table_id}`
    WHERE MMSI = 636014430
    """

df = client.query(querys).to_dataframe()
df

,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading
0,636014430,1673746588000000000,39.50607,-124.99619,5.2,288.1,288.0
1,636014430,1673766982000000000,39.75724,-125.65957,5.9,304.5,294.0
2,636014430,1673752122000000000,39.56346,-125.17134,6.3,265.3,289.0
3,636014430,1673733500000000000,39.30539,-124.63766,7.0,314.0,318.0
4,636014430,1674030453000000000,48.73388,-123.11670,11.8,62.7,65.0
...,...,...,...,...,...,...,...
18259,636014430,1673965532000000000,47.56520,-125.34216,11.1,356.2,354.0
18260,636014430,1673975913000000000,48.10313,-125.22443,11.1,17.5,19.0
18261,636014430,1673976321000000000,48.12329,-125.21491,11.1,18.8,19.0
18262,636014430,1673557414000000000,33.95796,-119.01822,11.1,302.5,302.0


In [9]:
#sample 100 rows from df and sort by BaseDateTime
df_sample = df.sample(n=100, random_state=42).sort_values(by='BaseDateTime')
df_sample.head()

,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading
5094,636014430,1672546365000000000,33.76495,-118.26010,0.0,100.9,176.0
7732,636014430,1672620459000000000,33.73685,-118.22349,0.1,35.5,289.0
4877,636014430,1672637742000000000,33.73619,-118.22399,0.0,340.1,326.0
8409,636014430,1672651603000000000,33.73655,-118.22365,0.2,219.4,298.0
4754,636014430,1672691030000000000,33.73764,-118.22559,0.0,124.2,147.0


In [10]:
# create a plot with folium with the lat and lon of the dataframe
import folium
m = folium.Map(location=[df_sample['LAT'].mean(), df_sample['LON'].mean()], zoom_start=10)
for index, row in df_sample.iterrows():
    folium.Marker(location=[row['LAT'], row['LON']], popup=row['MMSI']).add_to(m)       
m